# Fase 6.1 — Simulación prequential del componente adaptativo

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

En la v1 el "aprendizaje incremental" casi no podía verse: cada crédito real reemplazaba un registro entre ~180.000 de Home Credit y el peso de FINTECH quedaba fijo en 20 %. El resultado de "sin diferencia" era esperable por construcción.

Aquí reformulo la adaptación como **gobierno del modelo**, no como aprendizaje en línea:

1. **Estático (M2):** ajusto una vez y dejo los coeficientes congelados.
2. **Ventana expansiva (M2):** reajusto cada trimestre con todo lo ya conocido.
3. **Ponderación temporal:** igual que la expansiva, pero con pesos que decaen a la mitad cada 18 meses.
4. **Bayesiano secuencial (mecanismo de M3):** el *prior* se centra en el modelo vigente; τ crece con la raíz de los eventos propios, así que el ancla se afloja cuando hay evidencia nueva.

La regla de honestidad es la fecha de conocimiento: el desenlace de un crédito desembolsado en t0 solo entra al entrenamiento en **t0 + 13 meses** (12 de observación + 1 de demora operativa). No uso desenlaces futuros.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..", "src").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

from tesis_v2 import adaptativo as ad, config, contexto, figuras, io_datos

config.asegurar_carpetas()
figuras.estilo()
np.random.seed(config.SEMILLA)

D, ctx, ds = contexto.cargar()
print(f"Población primaria: {len(D)} créditos, {int(D['y'].sum())} eventos ({100*D['y'].mean():.2f} %)")
print(f"Home Credit (prior inicial de la estrategia bayesiana): {len(ctx.beta_hc)} coeficientes compartidos")
ctx.beta_hc.round(4)

Población primaria: 733 créditos, 23 eventos (3.14 %)
Home Credit (prior inicial de la estrategia bayesiana): 6 coeficientes compartidos


edad_t0                -0.0211
log_plazo              -0.1190
tiene_historia          0.7053
log_n_previos          -0.1434
meses_cliente          -0.0095
log_peor_mora_previa    0.1895
dtype: float64

## 1. Calendario de conocimiento

Antes de calificar un origen trimestral solo puedo entrenar con créditos cuyo desenlace ya maduró. Eso deja un hueco de 13 meses entre el desembolso y su uso en el modelo.

In [2]:
Dp = ad.preparar(D)
cal = (Dp.groupby("trimestre_t0")
         .agg(desembolsos=("y", "size"), eventos=("y", "sum"),
              conocimiento_min=("conocimiento", "min"), conocimiento_max=("conocimiento", "max"))
         .reset_index())
cal["eventos_acum"] = cal["eventos"].cumsum()
io_datos.guardar_resultado(cal, "fase6", "calendario_conocimiento")
cal

,trimestre_t0,desembolsos,eventos,conocimiento_min,conocimiento_max,eventos_acum
0,2022Q1,5,0,2023-02-15,2023-04-24,0
1,2022Q2,9,2,2023-05-04,2023-07-30,2
2,2022Q3,14,1,2023-08-07,2023-10-21,3
3,2022Q4,21,3,2023-11-04,2024-01-23,6
4,2023Q1,43,1,2024-02-10,2024-04-30,7
5,2023Q2,49,0,2024-05-05,2024-07-30,7
6,2023Q3,48,1,2024-08-05,2024-10-21,8
7,2023Q4,57,3,2024-11-05,2025-01-28,11
8,2024Q1,54,2,2025-02-03,2025-04-28,13
9,2024Q2,56,0,2025-05-02,2025-07-27,13


## 2. Corro las cuatro estrategias

En cada origen trimestral:

- construyo el entrenamiento con `conocimiento < inicio del trimestre`;
- actualizo según la regla de la estrategia (el estático solo la primera vez que hay ≥ 5 eventos);
- califico los desembolsos de ese trimestre;
- registro además si el PSI o el conteo de eventos *habrían* disparado un reentrenamiento.

In [3]:
preds, bitacora, estados = ad.simular_prequential(D, beta_hc=ctx.beta_hc, cada_eventos=10, ctx=ctx)
print("Predicciones:", len(preds), "filas;", preds.groupby("estrategia").size().to_dict())
print("Orígenes con al menos un evento en prueba:")
print(bitacora.groupby("origen")["eventos_test"].max().loc[lambda s: s > 0])
bitacora.head(8)

Predicciones: 2165 filas; {'bayes_secuencial': 433, 'estatico': 433, 'expansivo': 433, 'ponderado': 433, 'tasa_base': 433}
Orígenes con al menos un evento en prueba:
origen
2022Q2    2
2022Q3    1
2022Q4    3
2023Q1    1
2023Q3    1
2023Q4    3
2024Q1    2
2024Q3    3
2024Q4    2
2025Q1    3
2025Q2    1
2025Q3    1
Name: eventos_test, dtype: int64


,origen,estrategia,corte,n_train,eventos_train,n_test,eventos_test,actualizo,tau,eventos_nuevos,eventos_acumulados_desde_update,disparo_eventos,disparo_psi,psi_max,psi_puntaje
0,2022Q1,tasa_base,2022-01-01,0,0,5,0,False,NaN,0,0,False,False,0.0,NaN
1,2022Q1,estatico,2022-01-01,0,0,5,0,False,NaN,0,0,False,False,0.0,NaN
2,2022Q1,expansivo,2022-01-01,0,0,5,0,False,NaN,0,0,False,False,0.0,NaN
3,2022Q1,ponderado,2022-01-01,0,0,5,0,False,NaN,0,0,False,False,0.0,NaN
4,2022Q1,bayes_secuencial,2022-01-01,0,0,5,0,False,NaN,0,0,False,False,0.0,NaN
5,2022Q2,tasa_base,2022-04-01,0,0,9,2,False,NaN,0,0,False,False,0.0,NaN
6,2022Q2,estatico,2022-04-01,0,0,9,2,False,NaN,0,0,False,False,0.0,NaN
7,2022Q2,expansivo,2022-04-01,0,0,9,2,False,NaN,0,0,False,False,0.0,NaN


## 3. Desempeño agregado y por origen

Comparo las estrategias sobre los créditos que todas lograron calificar (los orígenes tempranos sin modelo aún no entran). El *bootstrap* es por cliente, pareado entre estrategias.

In [4]:
res = ad.metricas_por_estrategia(preds)
por_origen = ad.metricas_por_origen(preds)
io_datos.guardar_resultado(res, "fase6", "metricas_estrategias")
io_datos.guardar_resultado(por_origen, "fase6", "metricas_por_origen")
io_datos.guardar_resultado(bitacora, "fase6", "bitacora_actualizaciones")
res.round(4)

,estrategia,n,eventos,tasa,log_loss,brier,brier_escalado,auc,pd_media
0,bayes_secuencial,433,10,0.0231,0.1556,0.0311,-0.3773,0.5028,0.0373
1,estatico,433,10,0.0231,0.1336,0.0288,-0.2746,0.6338,0.0428
2,expansivo,433,10,0.0231,0.1257,0.0253,-0.1210,0.5300,0.0373
3,ponderado,433,10,0.0231,0.1256,0.0252,-0.1151,0.5225,0.0350
4,tasa_base,433,10,0.0231,0.1203,0.0235,-0.0427,0.4661,0.0480


In [5]:
# Comparaciones pareadas frente al estático (referencia natural de "no adaptar")
comps = []
for otra in ["tasa_base", "expansivo", "ponderado", "bayes_secuencial"]:
    c = ad.bootstrap_pareado(preds, otra, "estatico", B=2000)
    comps.append({
        "estrategia": otra, "frente_a": "estatico",
        "delta_log_loss": c["log_loss"]["delta"],
        "ll_ic_inf": c["log_loss"]["ic_inf"], "ll_ic_sup": c["log_loss"]["ic_sup"], "ll_p": c["log_loss"]["p"],
        "delta_auc": c["auc"]["delta"],
        "auc_ic_inf": c["auc"]["ic_inf"], "auc_ic_sup": c["auc"]["ic_sup"], "auc_p": c["auc"]["p"],
        "n": c["n_comunes"], "eventos": c["eventos"],
    })
# Bayesiano frente a expansivo (¿vale el prior secuencial frente a reajustar libre?)
c = ad.bootstrap_pareado(preds, "bayes_secuencial", "expansivo", B=2000)
comps.append({
    "estrategia": "bayes_secuencial", "frente_a": "expansivo",
    "delta_log_loss": c["log_loss"]["delta"],
    "ll_ic_inf": c["log_loss"]["ic_inf"], "ll_ic_sup": c["log_loss"]["ic_sup"], "ll_p": c["log_loss"]["p"],
    "delta_auc": c["auc"]["delta"],
    "auc_ic_inf": c["auc"]["ic_inf"], "auc_ic_sup": c["auc"]["ic_sup"], "auc_p": c["auc"]["p"],
    "n": c["n_comunes"], "eventos": c["eventos"],
})
comp = pd.DataFrame(comps)
io_datos.guardar_resultado(comp, "fase6", "comparaciones_pareadas")
comp.round(4)

,estrategia,frente_a,delta_log_loss,ll_ic_inf,ll_ic_sup,ll_p,delta_auc,auc_ic_inf,auc_ic_sup,auc_p,n,eventos
0,tasa_base,estatico,-0.0133,-0.0361,0.0066,0.221,-0.1677,-0.3904,0.0374,0.119,433,10
1,expansivo,estatico,-0.0079,-0.0220,0.0042,0.249,-0.1038,-0.2251,0.0305,0.123,433,10
2,ponderado,estatico,-0.0080,-0.0226,0.0054,0.274,-0.1113,-0.2396,0.0302,0.118,433,10
3,bayes_secuencial,estatico,0.0220,0.0006,0.0480,0.045,-0.1310,-0.2354,-0.0242,0.019,433,10
4,bayes_secuencial,expansivo,0.0299,0.0082,0.0591,0.001,-0.0272,-0.1770,0.0828,0.794,433,10


## 4. Figuras I01 e I02

I01 muestra la trayectoria del AUC por origen. I02 resume el Δ log loss frente al estático con su IC.

In [6]:
# I01 — AUC por origen (solo trimestres con al menos un evento)
fig, ax = plt.subplots(figsize=(9.5, 4.2))
colores = {"tasa_base": "#333333", "estatico": figuras.GRIS, "expansivo": figuras.AZUL,
           "ponderado": figuras.NARANJA, "bayes_secuencial": figuras.VERDE}
nombres = {"tasa_base": "Tasa base conocida", "estatico": "M2 estático", "expansivo": "M2 expansivo",
           "ponderado": "M2 ponderado", "bayes_secuencial": "Bayes secuencial (M3)"}
sub = por_origen[por_origen["eventos"] > 0].copy()
for est_nombre, g in sub.groupby("estrategia"):
    ax.plot(g["origen"], g["auc"], marker="o", ms=4, lw=1.4,
            color=colores[est_nombre], label=nombres[est_nombre])
ax.axhline(0.5, color="#bbbbbb", ls="--", lw=0.8)
ax.set_ylabel("AUC")
ax.set_xlabel("Origen (trimestre de desembolso)")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="lower left", fontsize=8)
ax.set_title("Desempeño prequential por origen (solo trimestres con evento)")
figuras.guardar(fig, "fase6", "I01", "auc_por_origen",
                "Ninguna estrategia sostiene un AUC estable por origen; con pocos eventos por trimestre el trazo es ruidoso.")

src/tesis_v2/figuras.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# I02 — Δ log loss frente al estático
fig, ax = plt.subplots(figsize=(7.2, 3.8))
plot = comp[comp["frente_a"] == "estatico"].copy()
ypos = np.arange(len(plot))
ax.axvline(0, color="#bbbbbb", lw=0.9)
ax.errorbar(plot["delta_log_loss"], ypos,
            xerr=[plot["delta_log_loss"] - plot["ll_ic_inf"], plot["ll_ic_sup"] - plot["delta_log_loss"]],
            fmt="o", color=figuras.AZUL, capsize=3)
ax.set_yticks(ypos, [nombres[e] for e in plot["estrategia"]])
ax.set_xlabel("Δ log loss frente a M2 estático (negativo = mejor)")
ax.set_title("Δ log loss frente al M2 congelado")
figuras.guardar(fig, "fase6", "I02", "delta_logloss_vs_estatico",
                "Reentrenar mejora el log loss frente al M2 congelado, pero ninguna estrategia supera a la tasa base conocida.")

src/tesis_v2/figuras.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Disparadores retrospectivos (PSI y conteo)

Aunque la simulación actualiza por calendario, registro cuándo el PSI de predictores clave o el conteo de eventos nuevos habrían exigido un reentrenamiento. Eso alimenta el protocolo de monitoreo.

In [8]:
disp = bitacora[bitacora["estrategia"] == "bayes_secuencial"].copy()
resumen_disp = pd.DataFrame({
    "origenes": [disp["origen"].nunique()],
    "veces_disparo_eventos": [int(disp["disparo_eventos"].sum())],
    "veces_disparo_psi": [int(disp["disparo_psi"].fillna(False).sum())],
    "psi_max_mediano": [disp["psi_max"].replace(0, np.nan).median()],
    "psi_max_p90": [disp["psi_max"].replace(0, np.nan).quantile(0.9)],
})
io_datos.guardar_resultado(resumen_disp, "fase6", "resumen_disparadores")
print(resumen_disp.to_string(index=False))
print("\nOrígenes con disparo por eventos o PSI:")
print(disp.loc[disp["disparo_eventos"] | disp["disparo_psi"].fillna(False),
               ["origen", "eventos_nuevos", "eventos_acumulados_desde_update", "psi_max", "psi_puntaje"]].to_string(index=False))

 origenes  veces_disparo_eventos  veces_disparo_psi  psi_max_mediano  psi_max_p90
       19                      0                  9           3.7746      5.11175

Orígenes con disparo por eventos o PSI:
origen  eventos_nuevos  eventos_acumulados_desde_update  psi_max  psi_puntaje
2024Q3               1                                1 3.774600     0.098094
2024Q4               1                                1 5.259295     0.182333
2025Q1               0                                0 5.074864     1.059895
2025Q2               5                                5 3.916596     0.847623
2025Q3               0                                0 3.261077     0.295129
2025Q4               2                                2 4.179463     0.276585
2026Q1               2                                2 3.251742     0.049104
2026Q2               3                                3 3.687170     0.310339
2026Q3               1                                1 2.636520     0.478243


In [9]:
# Evolución de τ en la estrategia bayesiana
tau_hist = bitacora[(bitacora["estrategia"] == "bayes_secuencial") & bitacora["tau"].notna()][
    ["origen", "eventos_train", "tau", "n_train"]].drop_duplicates("origen")
io_datos.guardar_resultado(tau_hist, "fase6", "evolucion_tau")
tau_hist.round(3)

,origen,eventos_train,tau,n_train
49,2024Q2,6,1.0,75
54,2024Q3,7,1.0,126
59,2024Q4,8,1.0,173
64,2025Q1,8,1.0,227
69,2025Q2,13,1.0,283
74,2025Q3,13,1.0,341
79,2025Q4,15,1.0,402
84,2026Q1,17,1.0,476
89,2026Q2,20,1.0,534
94,2026Q3,21,1.0,590


## 6. Lectura preliminar

Guardo las predicciones agregadas (sin llaves de cliente en `resultados/`; el detalle por crédito queda en `_privado/`) y dejo el cierre interpretativo para el notebook 16 y el informe.

In [10]:
# Agregados publicables; detalle por crédito solo en _privado
agg = preds.groupby(["estrategia", "origen"]).agg(
    n=("y", "size"), eventos=("y", "sum"), pd_media=("p", "mean"), y_media=("y", "mean")
).reset_index()
io_datos.guardar_resultado(agg, "fase6", "predicciones_agregadas_por_origen")

priv = preds.copy()
priv["t0"] = priv["t0"].astype(str)
priv["conocimiento"] = priv["conocimiento"].astype(str)
io_datos.guardar_privado(priv, "prequential_predicciones_fase6")

# Excel parcial (se completa en el notebook 16)
wb_path = config.RESULTADOS / "fase6" / "p7_v2_adaptativo.xlsx"
with pd.ExcelWriter(wb_path, engine="openpyxl") as xl:
    cal.to_excel(xl, sheet_name="calendario", index=False)
    res.to_excel(xl, sheet_name="metricas", index=False)
    por_origen.to_excel(xl, sheet_name="por_origen", index=False)
    comp.to_excel(xl, sheet_name="comparaciones", index=False)
    bitacora.to_excel(xl, sheet_name="bitacora", index=False)
    tau_hist.to_excel(xl, sheet_name="tau", index=False)
    resumen_disp.to_excel(xl, sheet_name="disparadores", index=False)

reg = io_datos.escribir_registro("15_simulacion_prequential", {
    "salidas": [
        "resultados/fase6/metricas_estrategias.csv",
        "resultados/fase6/comparaciones_pareadas.csv",
        "resultados/fase6/bitacora_actualizaciones.csv",
        "resultados/fase6/p7_v2_adaptativo.xlsx",
        "figuras/fase6/I01_auc_por_origen.png",
        "figuras/fase6/I02_delta_logloss_vs_estatico.png",
    ],
    "n_preds": len(preds),
    "estrategias": list(res["estrategia"]),
})
print("Registro:", reg)
print("Excel:", wb_path)

Registro: evidencias/registro_15_simulacion_prequential.json
Excel: resultados/fase6/p7_v2_adaptativo.xlsx
